# Iceland: correcting the ionosphere in a NISAR pair

Track 29, frame 35, 2026-07-22 / 2026-08-15.

**This notebook is a walkthrough, not a runnable demonstration.**  A pair needs roughly
55 GB of RSLC downloads, 300 GB of free scratch, a CUDA GPU and several hours.  The cells
below are the commands in the order they are run, with a note on what each produces; run
them in a terminal, in a working directory of their own.

Everything reads `pair.yaml` beside this notebook, which carries the track, frame, dates
and the per-pair fill and unwrapping settings.

## 1. Screening: which pair is worth the RSLC chain

The GSLC products are geocoded and cheap, so the question of whether a pair carries
ionospheric streaking is settled before any RSLC is downloaded.  `stack` accumulates the
interferometric and power sums for every date pair once; `streak-index` turns that into
R_FFT, the angular concentration of the coherence loss, and theta, its orientation in radar
coordinates.

In [ ]:
# one pass over the GSLC stack (~25 GB per date; the granules can be deleted afterwards)
!ionoNISAR download --track 29 --frame 35 --level GSLC --dates 20260616 20260710 20260722 20260815
!ionoNISAR stack    --track 29 --frame 35 --dates 20260616 20260710 20260722 20260815

# R_FFT and theta at a 480 m cell, both frequencies
!ionoNISAR streak-index --track 29 --frame 35 --dates 20260616 20260710 20260722 20260815 \
    --rebin 2 --out gslc_streak/streak_index.csv

`select-pair` ranks the table.  R_FFT says how concentrated the coherence loss is in one
orientation, not how much coherence was lost, so it is a screening statistic: pairs above
about 0.5 generally show pronounced streaking, and that value is an empirical guide rather
than a threshold.  The sign of `D_RA` is the direction the anisotropic gap fill should
stretch along.

In [ ]:
!ionoNISAR select-pair gslc_streak/streak_index.csv --freq A --top 5

## 2. Inputs for the chosen pair

`pair.yaml` names the pair from here on.  The DEM comes from Copernicus GLO-30, the map
grid is an empty raster that pins every later product to one lattice, and the glacier mask
comes from the Randolph Glacier Inventory 7.0 -- offsets over ice measure ice flow, not
registration, so they are excluded and filled.

In [ ]:
!ionoNISAR download --track 29 --frame 35 --level RSLC --dates 20260722 20260815
!ionoNISAR --config pair.yaml dem
!ionoNISAR --config pair.yaml grid
!ionoNISAR --config pair.yaml glacier-mask

# what is present, what is missing, and whether this machine can run the rest
!ionoNISAR --config pair.yaml check

### A note on the DEM check

`check` reports the DEM as not covering the whole frame polygon on this frame.  That is
expected and not fatal: Copernicus GLO-30 is a land product with no tiles over open ocean,
and the frame's seaward vertices fall outside it.  The chain runs; the phase over water is
masked out anyway.

## 3. Registration

One command covers geometric coregistration (rdr2geo into the reference, geo2rdr into the
secondary, resample), dense offset estimation with PyCuAmpcor, masking and gap filling of
the offset field, the azimuth rubbersheet, the decision on Doppler-dependent refocusing,
and the interferogram.

The ionosphere shifts targets along track, and a single shift per pixel only registers the
centre of the Doppler band.  Where the applied field changes quickly along track, each
sub-aperture needs a different shift, and `--refocus-km auto` sweeps the pierce-point
aperture and applies the refocus only where it buys coherence.

This is the expensive step: about 1.5 hours on one A100, most of it in the two dense
correlation passes and the resampling.

In [ ]:
!ionoNISAR --config pair.yaml coregister

Worth reading in the log: the `[coreg] check` residuals, which say whether the geometric
registration converged; the fill's anisotropy, which should be well above 1 for a
range-elongated field; and the `[refocus]` verdict, which reports the aperture chosen and
the coherence it gained, or says that there was nothing to refocus.

## 4. The ionospheric phase screen

The hybrid screen takes its long along-track wavelengths from the split-spectrum estimate
and its short ones from the azimuth offsets.  The two are complementary: the azimuth route
integrates along track, so its noise accumulates as a random walk and it drifts at long
wavelengths, while the split-spectrum solve integrates nothing but amplifies band-phase
noise by 1/|det|.

Asking for `hybrid` builds both parents first; neither is rebuilt if it is already on disk.

In [ ]:
!ionoNISAR --config pair.yaml screen --routes hybrid

## 5. Or the whole thing at once

In [ ]:
!ionoNISAR --config pair.yaml run --routes hybrid

## 6. What comes out

Under `outputs_unified/hybrid/`:

| file | what it is |
|---|---|
| `ifg_phase_geo_t029_20260722_20260815_rbcs.tif` | the corrected interferometric phase, geocoded |
| `ifg_coh_geo_t029_20260722_20260815_rbcs.tif` | its coherence |
| `iono_screen_t029_20260722_20260815.npz` | the hybrid screen on the offset lattice |
| `iono_screen_t029_20260722_20260815.tif` | the same screen, geocoded |

`outputs_unified/common/` holds the rungs the screen was applied to, so the correction can be
undone or compared: `_prerb` (coregistered only), `_rb` (+ azimuth rubbersheet) and `_rbc`
(+ carrier compensation).  `outputs_unified/offsets/` and `outputs_unified/split/` hold the two parent
screens, which the hybrid needs and which are worth looking at on their own.